# Pipeline B Notebook (Qwen3.5-9B Enhance + Mistral-7B Classify)

本 Notebook 将原始 Pipeline B 改为两阶段文本流水线：
1. Qwen3.5-9B 对 OCR 文本做语义增强（生成简洁描述）
2. Mistral-7B 基于增强描述做 Task4/Task6 判别

In [1]:
import json
import os
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List

import numpy as np
import torch
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

from dataset import create_data_splits
from pipeline_b import (
    MemeRow,
    evaluate_pipeline_b,
    generate_text_llm,
    load_rows,
    resolve_local_model_path,
    select_prototypes_task4,
    select_prototypes_task6,
)

In [2]:
@dataclass
class Config:
    data_dir: str = "/root/QWen/EXIST 2024 Memes Dataset"
    output_dir: str = "/root/QWen/outputs/pipeline_b_qwen_mistral"
    enhanced_cache_path: str = "/root/QWen/outputs/pipeline_b_qwen_mistral/enhanced_texts.jsonl"

    seed: int = 42
    force_split: bool = False

    enhancer_model_name: str = "Qwen/Qwen3.5-9B"
    classifier_model_name: str = "/root/autodl-tmp/mistral_models/7B-Instruct-v0.3"
    model_cache_dir: str = "/root/autodl-tmp/hf-cache"

    k_task4: int = 3
    k_task6: int = 2

    max_train_rows: int = 0
    max_test_rows: int = 0

    max_enhance_tokens: int = 96


cfg = Config()
random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)

print(cfg)

Config(data_dir='/root/QWen/EXIST 2024 Memes Dataset', output_dir='/root/QWen/outputs/pipeline_b_qwen_mistral', enhanced_cache_path='/root/QWen/outputs/pipeline_b_qwen_mistral/enhanced_texts.jsonl', seed=42, force_split=False, enhancer_model_name='Qwen/Qwen3.5-9B', classifier_model_name='/root/autodl-tmp/mistral_models/7B-Instruct-v0.3', model_cache_dir='/root/autodl-tmp/hf-cache', k_task4=3, k_task6=2, max_train_rows=0, max_test_rows=0, max_enhance_tokens=96)


In [3]:
def model_name_to_cache_folder(model_name: str) -> str:
    return "models--" + model_name.replace("/", "--")


def resolve_model_path_with_integrity(cache_dir: str, model_name: str) -> str:
    """Prefer complete local snapshot; fallback to model name for auto-download if incomplete."""
    p = Path(cache_dir)
    snapshots_dir = p / model_name_to_cache_folder(model_name) / "snapshots"
    if snapshots_dir.exists():
        candidates = [x for x in snapshots_dir.iterdir() if x.is_dir() and (x / "config.json").exists()]
        if candidates:
            candidates.sort(key=lambda x: x.stat().st_mtime, reverse=True)
            snapshot = candidates[0]

            index_file = snapshot / "model.safetensors.index.json"
            if index_file.exists():
                try:
                    with open(index_file, "r", encoding="utf-8") as f:
                        index_obj = json.load(f)
                    weight_map = index_obj.get("weight_map", {})
                    required_files = sorted(set(weight_map.values()))
                    missing_files = [name for name in required_files if not (snapshot / name).exists()]
                    if missing_files:
                        print(
                            f"[model-check] Incomplete local snapshot for {model_name}. "
                            f"Missing {len(missing_files)} shard(s), will fallback to HF download."
                        )
                        return model_name
                except Exception as e:
                    print(f"[model-check] Cannot validate snapshot for {model_name}: {e}. Fallback to model name.")
                    return model_name

            return str(snapshot)

    return resolve_local_model_path(cache_dir, model_name)


def build_enhance_prompt(lang: str, ocr: str) -> str:
    return (
        "You are improving meme text understanding for sexism detection.\n"
        "Given OCR text and language, write a concise semantic description of the meme message.\n"
        "Focus on social and gender implications, avoid visual speculation, and keep it to 1-2 sentences.\n"
        f"Language: {lang}\n"
        f"OCR: {ocr}\n"
        "Enhanced description:"
    )


def load_or_generate_enhanced_texts(
    rows: List[MemeRow],
    cache_path: str,
    model,
    tokenizer,
    max_new_tokens: int = 96,
) -> Dict[str, str]:
    cache: Dict[str, str] = {}

    if os.path.exists(cache_path):
        with open(cache_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                obj = json.loads(line)
                cache[str(obj["sample_id"])] = str(obj["enhanced_text"] )

    missing = [r for r in rows if r.sample_id not in cache]
    if not missing:
        return cache

    os.makedirs(os.path.dirname(cache_path), exist_ok=True)
    with open(cache_path, "a", encoding="utf-8") as fout:
        for r in tqdm(missing, desc="Enhancing OCR text"):
            prompt = build_enhance_prompt(lang=r.lang, ocr=r.text)
            enhanced = generate_text_llm(
                model=model,
                tokenizer=tokenizer,
                prompt=prompt,
                max_new_tokens=max_new_tokens,
            )
            cache[r.sample_id] = enhanced
            fout.write(
                json.dumps(
                    {"sample_id": r.sample_id, "enhanced_text": enhanced},
                    ensure_ascii=False,
                )
                + "\n"
            )

    return cache

In [4]:
splits = create_data_splits(cfg.data_dir, seed=cfg.seed, force_split=cfg.force_split)
train_ids = splits.get("train", [])
test_ids = splits.get("test", [])

train_rows = load_rows(cfg.data_dir, train_ids)
test_rows = load_rows(cfg.data_dir, test_ids)

if cfg.max_train_rows > 0:
    train_rows = train_rows[: cfg.max_train_rows]
if cfg.max_test_rows > 0:
    test_rows = test_rows[: cfg.max_test_rows]

print(f"[data] train/test rows: {len(train_rows)}/{len(test_rows)}")

proto_task4 = select_prototypes_task4(train_rows, k_per_class=cfg.k_task4)
proto_task6 = select_prototypes_task6(train_rows, k_per_label=cfg.k_task6)

all_proto_ids = set()
for rows in proto_task4.values():
    for r in rows:
        all_proto_ids.add(r.sample_id)
for rows in proto_task6.values():
    for r in rows:
        all_proto_ids.add(r.sample_id)

row_by_id = {r.sample_id: r for r in train_rows + test_rows}
proto_rows = [row_by_id[sid] for sid in all_proto_ids if sid in row_by_id]
desc_rows = proto_rows + test_rows

print(f"[data] prototype rows: {len(proto_rows)}")

Loading existing data splits from /root/QWen/EXIST 2024 Memes Dataset/data_splits.json
[data] train/test rows: 3235/405
[data] prototype rows: 15


In [5]:
# Optional: replace prototypes with curated high-consensus examples (3-5 each).
curated_path = "/root/QWen/outputs/high_consensus_fewshot_examples.json"
if os.path.exists(curated_path):
    with open(curated_path, "r", encoding="utf-8") as f:
        curated = json.load(f)

    row_by_id = {r.sample_id: r for r in train_rows + test_rows}

    def _rows_by_ids(ids):
        return [row_by_id[sid] for sid in ids if sid in row_by_id]

    yes_ids = [x["sample_id"] for x in curated["task4"]["YES"][:5]]
    no_ids = [x["sample_id"] for x in curated["task4"]["NO"][:5]]
    proto_task4 = {
        "YES": _rows_by_ids(yes_ids),
        "NO": _rows_by_ids(no_ids),
    }

    proto_task6 = {}
    for lab in proto_task6.keys() if isinstance(proto_task6, dict) else []:
        pass
    for lab in curated.get("task6", {}):
        ids = [x["sample_id"] for x in curated["task6"][lab][:5]]
        proto_task6[lab] = _rows_by_ids(ids)

    all_proto_ids = set()
    for rows in proto_task4.values():
        for r in rows:
            all_proto_ids.add(r.sample_id)
    for rows in proto_task6.values():
        for r in rows:
            all_proto_ids.add(r.sample_id)

    proto_rows = [row_by_id[sid] for sid in all_proto_ids if sid in row_by_id]
    desc_rows = proto_rows + test_rows

    print("[curated] loaded high-consensus examples")
    print(f"[curated] task4 YES/NO: {len(proto_task4['YES'])}/{len(proto_task4['NO'])}")
    print("[curated] task6 per label:", {k: len(v) for k, v in proto_task6.items()})
else:
    print(f"[curated] not found: {curated_path}")
    print("[curated] keep default prototype selection")

[curated] loaded high-consensus examples
[curated] task4 YES/NO: 5/5
[curated] task6 per label: {'IDEOLOGICAL-INEQUALITY': 5, 'STEREOTYPING-DOMINANCE': 5, 'OBJECTIFICATION': 5, 'SEXUAL-VIOLENCE': 5, 'MISOGYNY-NON-SEXUAL-VIOLENCE': 5}


In [6]:
if "resolve_model_path_with_integrity" not in globals():
    def resolve_model_path_with_integrity(cache_dir: str, model_name: str) -> str:
        return resolve_local_model_path(cache_dir, model_name)

enhancer_model_path = resolve_model_path_with_integrity(cfg.model_cache_dir, cfg.enhancer_model_name)
print(f"[enhancer] model path: {enhancer_model_path}")

enhancer_tokenizer = AutoTokenizer.from_pretrained(enhancer_model_path, cache_dir=cfg.model_cache_dir)
enhancer_model = AutoModelForCausalLM.from_pretrained(
    enhancer_model_path,
    cache_dir=cfg.model_cache_dir,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)
enhancer_model.eval()

enhanced_cache = load_or_generate_enhanced_texts(
    rows=desc_rows,
    cache_path=cfg.enhanced_cache_path,
    model=enhancer_model,
    tokenizer=enhancer_tokenizer,
    max_new_tokens=cfg.max_enhance_tokens,
)

del enhancer_model
del enhancer_tokenizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"[enhancer] cached descriptions: {len(enhanced_cache)}")

[enhancer] model path: /root/autodl-tmp/hf-cache/models--Qwen--Qwen3.5-9B/snapshots/c202236235762e1c871ad0ccb60c8ee5ba337b9a


`torch_dtype` is deprecated! Use `dtype` instead!
The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

Enhancing OCR text: 100%|██████████| 19/19 [00:52<00:00,  2.74s/it]

[enhancer] cached descriptions: 439


## Cell 8: Patch Note

从这里开始不修改前面单元格。
下面的 Cell 9 会覆盖分类阶段加载逻辑，包含本地快照完整性检查与自动回退下载。

In [ ]:
# Cell 9: Robust classifier stage (no edits to previous cells)
import json
import os

def _resolve_model_path_with_integrity(cache_dir: str, model_name: str) -> str:
    if os.path.isdir(model_name):
        return model_name

    snapshots_dir = os.path.join(
        cache_dir,
        "models--" + model_name.replace("/", "--"),
        "snapshots",
    )
    if os.path.isdir(snapshots_dir):
        candidates = []
        for d in os.listdir(snapshots_dir):
            p = os.path.join(snapshots_dir, d)
            if os.path.isdir(p) and os.path.isfile(os.path.join(p, "config.json")):
                candidates.append(p)
        if candidates:
            candidates.sort(key=lambda x: os.path.getmtime(x), reverse=True)
            snapshot = candidates[0]
            index_file = os.path.join(snapshot, "model.safetensors.index.json")
            if os.path.isfile(index_file):
                try:
                    with open(index_file, "r", encoding="utf-8") as f:
                        index_obj = json.load(f)
                    weight_map = index_obj.get("weight_map", {})
                    required_files = sorted(set(weight_map.values()))
                    missing_files = [
                        name for name in required_files if not os.path.isfile(os.path.join(snapshot, name))
                    ]
                    if missing_files:
                        print(
                            f"[model-check] Incomplete local snapshot for {model_name}. "
                            f"Missing {len(missing_files)} shard(s), fallback to HF download."
                        )
                        return model_name
                except Exception as e:
                    print(f"[model-check] Validation failed for {model_name}: {e}. Fallback to HF download.")
                    return model_name
            return snapshot
    return resolve_local_model_path(cache_dir, model_name)

proto_task4_descs = {
    "YES": [(r.sample_id, enhanced_cache.get(r.sample_id, "")) for r in proto_task4["YES"]],
    "NO": [(r.sample_id, enhanced_cache.get(r.sample_id, "")) for r in proto_task4["NO"]],
}
proto_task6_descs = {
    label: [(r.sample_id, enhanced_cache.get(r.sample_id, "")) for r in rows]
    for label, rows in proto_task6.items()
}

clf_model_path = _resolve_model_path_with_integrity(cfg.model_cache_dir, cfg.classifier_model_name)
print(f"[classifier] model path: {clf_model_path}")

clf_tokenizer = AutoTokenizer.from_pretrained(clf_model_path, cache_dir=cfg.model_cache_dir)
clf_model = AutoModelForCausalLM.from_pretrained(
    clf_model_path,
    cache_dir=cfg.model_cache_dir,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)
clf_model.eval()

# evaluate_pipeline_b in pipeline_b.py now supports hard-hard/soft-soft modes.
# Here we request both and print whichever keys exist to avoid KeyError.
metrics = evaluate_pipeline_b(
    test_rows=test_rows,
    desc_cache=enhanced_cache,
    clf_model=clf_model,
    clf_tokenizer=clf_tokenizer,
    proto_task4_descs=proto_task4_descs,
    proto_task6_descs=proto_task6_descs,
    output_path=cfg.output_dir,
    hierarchical_gate=True,
    eval_mode="both",
)

print("\n==== Pipeline B (Qwen3.5-9B + Mistral-7B) done ====")
if "hard_hard" in metrics:
    print(f"Task4 F1(binary): {metrics['hard_hard']['task4']['f1_binary']:.4f}")
    print(f"Task4 F1(macro):  {metrics['hard_hard']['task4']['f1_macro']:.4f}")
    print(f"Task6 F1(macro):  {metrics['hard_hard']['task6']['f1_macro']:.4f}")
    print(f"Task6 F1(micro):  {metrics['hard_hard']['task6']['f1_micro']:.4f}")
    print(f"Task6 F1(samples):{metrics['hard_hard']['task6']['f1_samples']:.4f}")
elif "task4" in metrics and "task6" in metrics:
    print(f"Task4 F1(binary): {metrics['task4']['f1_binary']:.4f}")
    print(f"Task4 F1(macro):  {metrics['task4']['f1_macro']:.4f}")
    print(f"Task6 F1(macro):  {metrics['task6']['f1_macro']:.4f}")
    print(f"Task6 F1(micro):  {metrics['task6']['f1_micro']:.4f}")
    print(f"Task6 F1(samples):{metrics['task6']['f1_samples']:.4f}")

if "soft_soft" in metrics:
    print(f"Task4 Brier:       {metrics['soft_soft']['task4']['brier']:.4f}")
    print(f"Task6 Brier(macro):{metrics['soft_soft']['task6']['brier_macro']:.4f}")
    print(f"Task6 Brier(micro):{metrics['soft_soft']['task6']['brier_micro']:.4f}")

print(f"samples: {metrics['num_samples']}")
print(f"metrics path: {os.path.join(cfg.output_dir, 'metrics.json')}")

[classifier] model path: /root/autodl-tmp/mistral_models/7B-Instruct-v0.3


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Classifying: 100%|██████████| 405/405 [03:30<00:00,  1.92it/s]



==== Pipeline B (Qwen3.5-9B + Mistral-7B) done ====


KeyError: 'task4'